# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fksifat/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated refresh model into a practical content-action playbook. It focuses on what a human should do first, why it should be done first, and where the model should stay in the background.

The language here stays honest: observed, measured, directional, and decision-support.


## 1. Ranked actions + reason codes

The ranked queue should be read as a triage list for refresh review. The top rows are the pages that look most worth a human check first because they combine visible demand, signs of freshness decay, and model-based decline risk. The reason codes are there to make that choice explainable: a good action is not just “high score,” but “high score because the page is visible, stale, and still attracting demand.”

In [1]:
from pathlib import Path
import json
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "work").exists() and (candidate / "skills").exists():
            return candidate
    return start

ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = ROOT / "work" / "outputs"
FIGURE_DIR = ROOT / "work" / "figures"
FIGURE_DIR.mkdir(exist_ok=True, parents=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

queue_path = ROOT / "outputs" / "refresh_queue.csv"
queue = pd.read_csv(queue_path)
queue = queue.sort_values("final_rank").reset_index(drop=True)

playbook = queue[[
    "final_rank",
    "content_id",
    "client_id",
    "final_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "best_model_probability",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "trend_direction",
]].copy()

playbook["reason_codes"] = playbook["final_reason_codes"].fillna("").str.split("|")
playbook["top_reason"] = playbook["reason_codes"].apply(lambda xs: xs[0] if xs else "general_refresh_review")
playbook["action_label"] = playbook["suggested_action"].replace({
    "refresh_and_review_ctr": "Refresh and review CTR",
    "refresh_and_review_engagement": "Refresh and review engagement",
    "refresh": "Refresh",
    "monitor": "Monitor",
    "expand_and_refresh": "Expand and refresh",
})

print(f"Repo root: {ROOT}")
print(f"Loaded {len(playbook):,} ranked rows from {queue_path}")
print(playbook[["final_rank", "confidence", "action_label", "top_reason"]].head(10).to_string(index=False))

Repo root: /home/farhankabirsifat/Desktop/flyrank-ml-internship
Loaded 30,000 ranked rows from /home/farhankabirsifat/Desktop/flyrank-ml-internship/outputs/refresh_queue.csv
 final_rank confidence           action_label            top_reason
          1       high Refresh and review CTR declining_with_demand
          2       high Refresh and review CTR declining_with_demand
          3     medium Refresh and review CTR declining_with_demand
          4     medium Refresh and review CTR declining_with_demand
          5       high Refresh and review CTR declining_with_demand
          6       high Refresh and review CTR declining_with_demand
          7       high Refresh and review CTR declining_with_demand
          8     medium                Refresh declining_with_demand
          9       high Refresh and review CTR declining_with_demand
         10       high Refresh and review CTR declining_with_demand


## 2. Intended use and limits

This playbook is intended for content operations and editorial review. It helps a team decide which pages deserve a closer look first, especially when a page is still visible in search but looks like it may be drifting or losing relevance. It is not a production automation engine and it should not be treated as a causal decision rule. The evidence is directional and based on a historical dataset, not a controlled experiment.

In [2]:
# Summarize the queue into a simple action playbook narrative.
summary = {
    "rows_scored": int(len(queue)),
    "high_confidence_rows": int((queue["confidence"] == "high").sum()),
    "medium_confidence_rows": int((queue["confidence"] == "medium").sum()),
    "low_confidence_rows": int((queue["confidence"] == "low").sum()),
    "top_actions": queue["suggested_action"].value_counts().to_dict(),
    "top_reason_codes": queue["final_reason_codes"].astype(str).str.split("|").explode().value_counts().head(10).to_dict(),
    "top_ranked_example": queue.loc[0, ["content_id", "best_model_probability", "final_refresh_score", "suggested_action", "final_reason_codes"]].to_dict(),
}

print(json.dumps(summary, indent=2))


{
  "rows_scored": 30000,
  "high_confidence_rows": 3576,
  "medium_confidence_rows": 11424,
  "low_confidence_rows": 15000,
  "top_actions": {
    "monitor": 13069,
    "refresh": 8207,
    "refresh_and_review_ctr": 6655,
    "refresh_and_review_engagement": 1987,
    "expand_and_refresh": 82
  },
  "top_reason_codes": {
    "declining_with_demand": 13152,
    "visible_model_opportunity": 10653,
    "low_ctr_visible_page": 9759,
    "ctr_review_candidate": 9759,
    "general_refresh_review": 9117,
    "model_decline_risk": 8459,
    "page_one_decay_risk": 7076,
    "low_engagement_visible_page": 6508,
    "engagement_review_candidate": 6508,
    "thin_visible_page": 82
  },
  "top_ranked_example": {
    "content_id": "content_1f080331fa2b",
    "best_model_probability": 0.7862466719906577,
    "final_refresh_score": 81.92846726727653,
    "suggested_action": "refresh_and_review_ctr",
    "final_reason_codes": "declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|mode

## 3. Human review + the no-go list

A person should review the page's current business importance, recent editorial changes, and whether a refresh would still align with the product or campaign plan. The queue is useful for prioritization, but humans should decide whether to refresh, leave alone, or hold for a different moment. The no-go list should include campaign-critical pages, policy-sensitive content, and low-signal pages where the model has little evidence to stand on.

In [3]:
# Human review rules and no-go cases.
review_rules = [
    "Treat the ranked queue as a starting point for review, not an automated dispatch list.",
    "Have a person check the content's current business value, recent updates, and whether a refresh would still be appropriate.",
    "Do not auto-act on items with low confidence, weak evidence, or sensitive or policy-risk content.",
    "Do not automate refreshes for brand-new, time-sensitive, or campaign-critical pages without human sign-off.",
]

no_go_cases = [
    "Never automate a refresh for pages that are actively in a live campaign or legal review.",
    "Do not rely on the queue for pages with very low impressions or tiny traffic volume.",
    "Do not use the queue as a causal decision rule; it is a prioritization aid.",
]

print("Review rules:")
for rule in review_rules:
    print(f"- {rule}")
print("\nNo-go cases:")
for item in no_go_cases:
    print(f"- {item}")


Review rules:
- Treat the ranked queue as a starting point for review, not an automated dispatch list.
- Have a person check the content's current business value, recent updates, and whether a refresh would still be appropriate.
- Do not auto-act on items with low confidence, weak evidence, or sensitive or policy-risk content.
- Do not automate refreshes for brand-new, time-sensitive, or campaign-critical pages without human sign-off.

No-go cases:
- Never automate a refresh for pages that are actively in a live campaign or legal review.
- Do not rely on the queue for pages with very low impressions or tiny traffic volume.
- Do not use the queue as a causal decision rule; it is a prioritization aid.


## 4. Monitoring / retrain triggers

The playbook should be monitored if the top recommendations stop matching what the team sees in real traffic, if the share of high-confidence actions drops sharply, or if the model starts surfacing the same kind of pages without useful signal. A retraining or review cycle is also worth doing when the underlying feature mix changes, the freshness signals shift, or the team starts ignoring the queue in practice.

In [4]:
# Monitoring and retrain triggers.
monitoring_triggers = [
    "If the queue's top-ranked actions stop matching new traffic patterns for 2-3 weeks, the scoring logic may be stale.",
    "If the share of high-confidence items drops sharply, re-check the feature windows and label logic.",
    "If the model's top feature importances shift materially, review the underlying freshness and visibility signals.",
    "If refresh actions are being ignored or overused by the team, update the playbook thresholds and review prompts.",
]

print("Monitoring and retrain triggers:")
for trigger in monitoring_triggers:
    print(f"- {trigger}")


Monitoring and retrain triggers:
- If the queue's top-ranked actions stop matching new traffic patterns for 2-3 weeks, the scoring logic may be stale.
- If the share of high-confidence items drops sharply, re-check the feature windows and label logic.
- If the model's top feature importances shift materially, review the underlying freshness and visibility signals.
- If refresh actions are being ignored or overused by the team, update the playbook thresholds and review prompts.


## 5. Exports for the paper

The ranked queue and a compact summary are exported for the paper so the next step can build on a consistent artifact. The exports are meant to be receipts: they preserve the ranked order, the action labels, and the rationale behind each recommendation without turning the queue into a black box.

In [5]:
# Export ranked queue and supporting artifacts for the paper.
playbook_export = playbook.copy()
playbook_export.to_csv(OUTPUT_DIR / "action_playbook_queue.csv", index=False)

with open(OUTPUT_DIR / "action_playbook_summary.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)

md_lines = [
    "# Action playbook summary",
    "",
    f"- Rows ranked: {len(playbook_export):,}",
    f"- High confidence: {int((queue['confidence'] == 'high').sum()):,}",
    f"- Medium confidence: {int((queue['confidence'] == 'medium').sum()):,}",
    f"- Low confidence: {int((queue['confidence'] == 'low').sum()):,}",
    "- Intended use: prioritize refresh review for likely declining or stale-visible pages.",
    "- Not for automation: campaign-critical, policy-sensitive, or low-signal pages.",
    "",
    "## Top actions",
]
for action, count in queue["suggested_action"].value_counts().items():
    md_lines.append(f"- {action}: {int(count):,}")

(OUTPUT_DIR / "action_playbook_summary.md").write_text("\n".join(md_lines), encoding="utf-8")

print(f"Wrote {OUTPUT_DIR / 'action_playbook_queue.csv'}")
print(f"Wrote {OUTPUT_DIR / 'action_playbook_summary.json'}")
print(f"Wrote {OUTPUT_DIR / 'action_playbook_summary.md'}")


Wrote /home/farhankabirsifat/Desktop/flyrank-ml-internship/work/outputs/action_playbook_queue.csv
Wrote /home/farhankabirsifat/Desktop/flyrank-ml-internship/work/outputs/action_playbook_summary.json
Wrote /home/farhankabirsifat/Desktop/flyrank-ml-internship/work/outputs/action_playbook_summary.md


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors in this environment
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The exports were generated under `work/outputs/` for the paper workflow